In [ ]:
from math import log

import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from networkx.drawing import circular_layout
from typing import Optional

import matplotlib.pyplot as plt
from matplotlib.path import Path as MplPath
import matplotlib.patches as patches
import networkx as nx
import numpy as np

from os import listdir
from os.path import isfile, join

from pygments.lexers import math
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

In [ ]:
def node_match_traditional(node1, node2):
    return node1 == node2

def node_subst_cost_traditional(node1, node2):
    return 1

def node_ins_del_cost_traditional(node):
    return 1

def edge_match_traditional(edge1, edge2):
    return edge1['motion_type'] == edge2['motion_type']

def edge_subst_cost_traditional(edge1, edge2):
    return 1+abs(edge1['all_DOFs'] - edge2['all_DOFs'])


def edge_ins_del_cost_traditional(edge):
    return edge['all_DOFs']


def get_traditional_similarity(G1, G2):
    distance = nx.graph_edit_distance(

        G1,
        G2,
        node_match=node_match_traditional,
        node_subst_cost=node_subst_cost_traditional,
        node_del_cost=node_ins_del_cost_traditional,
        node_ins_cost=node_ins_del_cost_traditional,
        edge_match=edge_match_traditional,
        edge_subst_cost=edge_subst_cost_traditional,
        edge_del_cost=edge_ins_del_cost_traditional,
        edge_ins_cost=edge_ins_del_cost_traditional,
        timeout=3,
    )

    return distance


def node_match_text_labels_MAG(node1, node2):

    return (
            node1['motion_type'] == node2['motion_type']
            and
            node1['softness'] == node2['softness']
    )


def node_subst_cost_text_labels_MAG(node1, node2):

    return (
        1
        +
        abs(node1['all_DOFs'] - node2['all_DOFs'])
        +
        abs(node1['softness'] - node2['softness'])
        )


def node_ins_del_cost_text_labels_MAG(node):
    return (
        node['all_DOFs'] + node['softness']
        )

def edge_match_text_labels_MAG(edge1, edge2):
    return edge1 == edge2

def edge_subst_cost_text_labels_MAG(edge1, edge2):
    return 1

def edge_ins_del_cost_text_labels_MAG(edge):
    return 1

def get_text_labels_MAG_similarity(G1, G2):
    distance = nx.graph_edit_distance(

        G1,
        G2,
        node_match=node_match_text_labels_MAG,
        node_subst_cost=node_subst_cost_text_labels_MAG,
        node_del_cost=node_ins_del_cost_text_labels_MAG,
        node_ins_cost=node_ins_del_cost_text_labels_MAG,
        edge_match=edge_match_text_labels_MAG,
        edge_subst_cost=edge_subst_cost_text_labels_MAG,
        edge_del_cost=edge_ins_del_cost_text_labels_MAG,
        edge_ins_cost=edge_ins_del_cost_text_labels_MAG,
        timeout=3,
    )

    return distance

In [ ]:
def graph_repeated_extension(list_nodes, list_edges):

    dict_nodes_repeated = dict()
    updated_list_nodes = []
    updated_list_edges = []

    for node_ in list_nodes:
        node = list(node_)
        node_name = str(node[0])
        node_name_len = len(node[0])
        number_node_instances = node[1]['symmetric_instances']
        dict_nodes_repeated[node_name] = number_node_instances
        for instance in range(int(number_node_instances)):

            node[0] = node_name[:node_name_len] + str(1+instance)
            new_node = node.copy()
            updated_list_nodes.append(new_node)

    in_list_edges_upd = []

    for edge_ in list_edges:
        edge = list(edge_)

        len_in_edge_name = len(edge[0])
        if edge[0] in dict_nodes_repeated:
            for instance_number in range(int(dict_nodes_repeated[edge[0]])):

                new_edge_name = str(edge[0])[:len_in_edge_name] + str(1+instance_number)
                edge[0] = new_edge_name
                new_edge = edge.copy()
                in_list_edges_upd.append(new_edge)


    for edge in in_list_edges_upd:
        edge = list(edge)
        len_out_edge_name = len(edge[1])
        if edge[1] in dict_nodes_repeated:
            for instance_number in range(int(dict_nodes_repeated[edge[1]])):

                    new_edge_name = str(edge[1])[:len_out_edge_name] + str(1+instance_number)
                    edge[1] = new_edge_name
                    new_edge = edge.copy()
                    updated_list_edges.append(new_edge)

    return updated_list_nodes, updated_list_edges

def graph_repeated_independent_extension(list_nodes, list_edges):

    dict_nodes_repeated = dict()
    updated_list_nodes = []
    updated_list_edges = []

    for node_ in list_nodes:
        node = list(node_)
        node_name = str(node[0])
        node_name_len = len(node[0])
        number_node_instances = node[1]['symmetric_instances']
        dict_nodes_repeated[node_name] = number_node_instances
        for instance in range(int(number_node_instances)):

            node[0] = node_name[:node_name_len] + str(1+instance)
            new_node = node.copy()
            updated_list_nodes.append(new_node)

    in_list_edges_upd = []

    for edge_ in list_edges:
        edge = list(edge_)

        len_in_edge_name = len(edge[0])
        if edge[0] in dict_nodes_repeated:
            for instance_number in range(int(dict_nodes_repeated[edge[0]])):
                if edge[0] != edge[1]:
                    new_edge_name = str(edge[0])[:len_in_edge_name] + str(1+instance_number)
                    edge[0] = new_edge_name
                    new_edge = edge.copy()
                    in_list_edges_upd.append(new_edge)
                else:
                    edge[0] = str(edge[0])[:len_in_edge_name] + str(1+instance_number)
                    edge[1] = str(edge[1])[:len_in_edge_name] + str(1+instance_number)
                    new_edge = edge.copy()
                    updated_list_edges.append(new_edge)

    for edge in in_list_edges_upd:
        edge = list(edge)
        len_out_edge_name = len(edge[1])
        if edge[1] in dict_nodes_repeated:
            for instance_number in range(int(dict_nodes_repeated[edge[1]])):
                if edge[0] != edge[1]:
                    new_edge_name = str(edge[1])[:len_out_edge_name] + str(1+instance_number)
                    edge[1] = new_edge_name
                    new_edge = edge.copy()
                    updated_list_edges.append(new_edge)
                else:
                    edge[0] = str(edge[0])[:len_out_edge_name] + str(1+instance_number)
                    edge[1] = str(edge[1])[:len_out_edge_name] + str(1+instance_number)
                    new_edge = edge.copy()
                    updated_list_edges.append(new_edge)

    return updated_list_nodes, updated_list_edges

In [ ]:
set_rigid_stimuli = {'ashtray_14n.jpg', 'axe_14n.jpg', 'boat_14n.jpg', 'bobsled_14s.jpg', 'candelabra_14s.jpg', 'coat_rack_13s.jpg', 'cufflink_16s.jpg', 'drain_16s.jpg', 'grate_13s.jpg', 'horseshoe_13n.jpg', 'hovercraft_13n.jpg', 'key_17s.jpg', 'television_14n.jpg', 'microscope_14n.jpg', }

set_soft_stimuli = {'beachball_16s.jpg',  'kimono_14s.jpg', 'lasagna_13s.jpg', 'mosquito_net_13s.jpg', 't-shirt_13s.jpg', 'tent_14n.jpg', 'whip_14s.jpg', 'wig_13s.jpg', 'hula_hoop_16s.jpg', 'dough_19s.jpg', 'nest_13s.jpg', 'chest1_14s.jpg',}

set_complex_stimuli = {'bike_14s.jpg', 'blind_19s.jpg', 'bulldozer_22n.jpg', 'crank_16s.jpg', 'helicopter_25s.jpg', 'joystick_14s.jpg', 'piano_13n.jpg', 'typewriter_13s.jpg', 'easel_15s.jpg', 'drawer_13s.jpg', 'watch_13s.jpg',}

set_complex_soft_non_animal_stimuli = {'umbrella_13s.jpg', 'pacifier_14s.jpg', 'clipboard_14s.jpg', 'headlamp_14s.jpg', 'banana_13s.jpg', 'pear_13s.jpg', 'quill_15s.jpg', 'brace_14s.jpg', 'dress_13s.jpg', }

set_animal_stimuli = {'alligator_14n.jpg', 'beaver_13n.jpg', 'butterfly_16n.jpg', 'chipmunk_16n.jpg', 'cow_16n.jpg', 'dragonfly_13s.jpg', 'hippopotamus_16s.jpg', 'horse_17s.jpg', 'iguana_20s.jpg', 'monkey_18n.jpg', 'rabbit_13n.jpg', 'starfish_15n.jpg', 'wasp_15n.jpg',}

list_ext = list(set_rigid_stimuli | set_soft_stimuli | set_complex_stimuli | set_complex_soft_non_animal_stimuli | set_animal_stimuli)
len(list_ext)

In [ ]:
def get_motion_traditional_graph(category_name):
    G = nx.Graph()
    G.name = category_name

    if category_name in set_rigid_stimuli or category_name in set_soft_stimuli or category_name in set_complex_soft_non_animal_stimuli:
        list_nodes = [
        ('base', {'symmetric_instances':1}),
        ]
        list_edges = [
        ]


    elif category_name in set_animal_stimuli:

        if category_name == 'starfish_15n.jpg':
            list_nodes = [
            ('tentacle', {'symmetric_instances':5}), ('body', {'symmetric_instances':1}), ('mouth', {'symmetric_instances':1})
            ]
            list_edges = [
                ('body', 'tentacle', {'motion_type':'2SO2','all_DOFs':2}),
                ('body', 'mouth', {'motion_type':'R','all_DOFs':1}),
            ]

        if category_name == 'alligator_14n.jpg':
            list_nodes = [

                 ('body', {'symmetric_instances':1}),
                 ('limb', {'symmetric_instances':4}),
                 ('tail', {'symmetric_instances':1}),
                 ('head', {'symmetric_instances':1}),
                 ('jaws', {'symmetric_instances':2}),

                ]

            list_edges = [

                ('head', 'jaws', {'motion_type':'SO2','all_DOFs':1}),
                ('head', 'body', {'motion_type':'SO3','all_DOFs':3}),
                ('body', 'limb', {'motion_type':'2SO2','all_DOFs':2}),
                ('body', 'tail', {'motion_type':'SO2','all_DOFs':1}),

                ]
        elif category_name == 'beaver_13n.jpg':

            list_nodes = [

                 ('body', {'symmetric_instances':1}),
                 ('arm',  {'symmetric_instances':2}),
                 ('leg',  {'symmetric_instances':2}),
                 ('tail', {'symmetric_instances':1}),
                 ('head', {'symmetric_instances':1}),

                ]

            list_edges = [

                ('head', 'body', {'motion_type':'SO3','all_DOFs':3}),
                ('body', 'arm', {'motion_type':'2SO2','all_DOFs':2}),
                ('body', 'leg', {'motion_type':'2SO2','all_DOFs':2}),
                ('body', 'tail', {'motion_type':'2SO2','all_DOFs':2}),

                ]

        elif category_name == 'cow_16n.jpg':

            list_nodes = [

                 ('body', {'symmetric_instances':1}),
                 ('thigh', {'symmetric_instances':4}),
                 ('calf', {'symmetric_instances':4}),
                 ('tail', {'symmetric_instances':1}),
                 ('head', {'symmetric_instances':1}),

                ]

            list_edges = [

                ('head', 'body', {'motion_type':'SO3','all_DOFs':3}),
                ('body', 'limb', {'motion_type':'2SO2','all_DOFs':2}),
                ('body', 'thigh', {'motion_type':'2SO2','all_DOFs':2}),
                ('calf', 'thigh', {'motion_type':'SO2','all_DOFs':1}),
                ('body', 'tail', {'motion_type':'2SO2','all_DOFs':2}),

                ]

        elif category_name == 'chipmunk_16n.jpg':

            list_nodes = [

                 ('body', {'symmetric_instances':1}),
                 ('limb', {'symmetric_instances':4}),
                 ('tail', {'symmetric_instances':1}),
                 ('head', {'symmetric_instances':1}),

                ]

            list_edges = [

                ('head', 'jaws', {'motion_type':'SO2','all_DOFs':1}),
                ('head', 'body', {'motion_type':'SO3','all_DOFs':3}),
                ('body', 'limb', {'motion_type':'2SO2','all_DOFs':2}),
                ('body', 'tail', {'motion_type':'2SO2','all_DOFs':2}),

                ]

        elif category_name == 'hippopotamus_16s.jpg':
            list_nodes = [

                 ('body', {'symmetric_instances':1}),
                 ('limb', {'symmetric_instances':4}),
                 ('head', {'symmetric_instances':1}),
                 ('jaws', {'symmetric_instances':2}),

                ]

            list_edges = [

                ('head', 'jaws', {'motion_type':'SO2','all_DOFs':1}),
                ('head', 'body', {'motion_type':'SO3','all_DOFs':3}),
                ('body', 'limb', {'motion_type':'2SO2','all_DOFs':2}),

                ]


        elif category_name == 'horse_17s.jpg':
            list_nodes = [

                 ('body', {'symmetric_instances':1}),
                 ('thigh', {'symmetric_instances':4}),
                 ('calf', {'symmetric_instances':4}),
                 ('tail', {'symmetric_instances':1}),
                 ('head', {'symmetric_instances':1}),

                ]

            list_edges = [

                ('head', 'body', {'motion_type':'SO3','all_DOFs':3}),
                ('body', 'limb', {'motion_type':'2SO2','all_DOFs':2}),
                ('body', 'thigh', {'motion_type':'2SO2','all_DOFs':2}),
                ('calf', 'thigh', {'motion_type':'SO2','all_DOFs':1}),
                ('body', 'tail', {'motion_type':'2SO2','all_DOFs':2}),

                ]

        elif category_name == 'iguana_20s.jpg':
            list_nodes = [

                 ('body', {'symmetric_instances':1}),
                 ('limb', {'symmetric_instances':4}),
                 ('tail', {'symmetric_instances':1}),
                 ('head', {'symmetric_instances':1}),
                 ('jaws', {'symmetric_instances':2}),
                 ('spikes', {'symmetric_instances':5}),

                ]

            list_edges = [

                ('head', 'body', {'motion_type':'SO3','all_DOFs':3}),
                ('body', 'limb', {'motion_type':'2SO2','all_DOFs':2}),
                ('body', 'tail', {'motion_type':'2SO2','all_DOFs':2}),
                ('body', 'spikes', {'motion_type':'2SO2','all_DOFs':2}),

                ]

        elif category_name == 'monkey_18n.jpg':
            list_nodes = [

                 ('body', {'symmetric_instances':1}),
                 ('forearm',  {'symmetric_instances':2}),
                 ('humerus',  {'symmetric_instances':2}),
                 ('wrist',  {'symmetric_instances':2}),
                 ('thigh',  {'symmetric_instances':2}),
                 ('calf',  {'symmetric_instances':2}),
                 ('tail', {'symmetric_instances':1}),
                 ('head', {'symmetric_instances':1}),

                ]
            list_edges = [

                ('head', 'body', {'motion_type':'SO3','all_DOFs':3}),
                ('body', 'tail', {'motion_type':'2SO2','all_DOFs':2}),
                ('body', 'thigh', {'motion_type':'2SO2','all_DOFs':2}),
                ('thigh', 'calf', {'motion_type':'2SO2','all_DOFs':2}),
                ('body', 'forearm', {'motion_type':'2SO2','all_DOFs':2}),
                ('forearm', 'humerus', {'motion_type':'SO2','all_DOFs':1}),
                ('wrist', 'humerus', {'motion_type':'SO3','all_DOFs':3}),

                ]
        elif category_name == 'rabbit_13n.jpg':
            list_nodes = [

                 ('body', {'symmetric_instances':1}),
                 ('arm',  {'symmetric_instances':2}),
                 ('leg',  {'symmetric_instances':2}),
                 ('head', {'symmetric_instances':1}),

                ]
            list_edges = [

                ('head', 'body', {'motion_type':'SO3','all_DOFs':3}),
                ('body', 'arm', {'motion_type':'2SO2','all_DOFs':2}),
                ('body', 'leg', {'motion_type':'2SO2','all_DOFs':2}),

                ]
        elif category_name == 'butterfly_16n.jpg':


            list_nodes = [

                 ('body', {'symmetric_instances':1}),
                 ('limbs',  {'symmetric_instances':5}),
                 ('wings',  {'symmetric_instances':2}),

                ]
            list_edges = [

                ('body', 'wings', {'motion_type':'SO2','all_DOFs':1}),
                ('body', 'limbs', {'motion_type':'2SO2','all_DOFs':2}),

                ]


        elif category_name == 'dragonfly_13s.jpg':

            list_nodes = [

                 ('body', {'symmetric_instances':1}),
                 ('limbs',  {'symmetric_instances':5}),
                 ('wings',  {'symmetric_instances':4}),

                ]
            list_edges = [

                ('body', 'wings', {'motion_type':'SO2','all_DOFs':1}),
                ('body', 'limbs', {'motion_type':'2SO2','all_DOFs':2}),

                ]
        elif category_name == 'wasp_15n.jpg':
            list_nodes = [

                 ('body', {'symmetric_instances':1}),
                 ('limbs',  {'symmetric_instances':5}),
                 ('wings',  {'symmetric_instances':2}),

                ]
            list_edges = [

                ('body', 'wings', {'motion_type':'SO2','all_DOFs':1}),
                ('body', 'limbs', {'motion_type':'2SO2','all_DOFs':2}),

                ]
    elif category_name in set_complex_stimuli:

        if category_name == 'crank_16s.jpg':
            list_nodes = [
                ('handle',  {'symmetric_instances':1}),
                ('arm', {'symmetric_instances':1}),
                ('base', {'symmetric_instances':1}),
            ]
            list_edges = [
                ('base', 'arm', {'motion_type':'SO2','all_DOFs':1}),
                ('arm', 'handle', {'motion_type':'SO2','all_DOFs':1}),
            ]
        elif category_name == 'drawer_13s.jpg':
            list_nodes = [
                ('drawer', {'symmetric_instances':3}),
                ('base', {'symmetric_instances':1}),
            ]
            list_edges = [
                ('base', 'drawer', {'motion_type':'R','all_DOFs':1}),
            ]


        elif category_name == 'easel_15s.jpg':
            list_nodes = [
                ('base', {'symmetric_instances':1}),
                ('leg', {'symmetric_instances':1}),
            ]
            list_edges = [
                ('base', 'leg', {'motion_type':'SO2','all_DOFs':1}),

            ]

        elif category_name == 'helicopter_25s.jpg':
            list_nodes = [
                ('blade1', {'symmetric_instances':1}),
                ('blade2', {'symmetric_instances':1}),
                ('frame', {'symmetric_instances':1}),
            ]
            list_edges = [
                ('blade1', 'frame', {'motion_type':'SO2','all_DOFs':1}),
                ('blade2', 'frame', {'motion_type':'SO2','all_DOFs':1}),
            ]

        elif category_name == 'joystick_14s.jpg':
            list_nodes = [
                ('base', {'symmetric_instances':1}),
                ('stick', {'symmetric_instances':1}),
                ('button', {'symmetric_instances':2}),
            ]
            list_edges = [
                ('base', 'stick', {'motion_type':'2SO2','all_DOFs':2}),
                ('base', 'button', {'motion_type':'R','all_DOFs':1}),
            ]

        elif category_name == 'typewriter_13s.jpg':
            list_nodes = [
                ('base', {'symmetric_instances':1}),
                ('key',{'symmetric_instances':5}),
            ]
            list_edges = [
                ('base', 'key', {'motion_type':'R','all_DOFs':1}),
            ]

        elif category_name == 'piano_13n.jpg':
            list_nodes = [
                ('key', {'symmetric_instances':5}),
                ('base', {'symmetric_instances':1}),
                ('lid', {'symmetric_instances':1}),
            ]
            list_edges = [
                ('base', 'key', {'motion_type':'R','all_DOFs':1}),
                ('base', 'lid', {'motion_type':'SO2','all_DOFs':1}),
            ]

        elif category_name == 'bike_14s.jpg':
            list_nodes = [
                ('base', {'symmetric_instances':1}),
                ('wheel', {'symmetric_instances':2}),
                ('pedals', {'symmetric_instances':1}),
            ]
            list_edges = [
                ('base', 'pedals', {'motion_type':'SO2','all_DOFs':1}),
                ('base', 'wheel', {'motion_type':'SO2+R','all_DOFs':2}),
            ]

        elif category_name == 'blind_19s.jpg':
            list_nodes = [
                ('base', {'symmetric_instances':1}),
                ('blind', {'symmetric_instances':1}),
            ]
            list_edges = [
                ('base', 'blind', {'motion_type':'R','all_DOFs':1}),
            ]


        elif category_name == 'bulldozer_22n.jpg':
            list_nodes = [
                ('base', {'symmetric_instances':1}),
                ('links', {'symmetric_instances':2}),
            ]
            list_edges = [
                ('base', 'links', {'motion_type':'SO2+R','all_DOFs':1}),
            ]

        elif category_name in ['watch_13s.jpg', ]:

            list_nodes = [
                ('base', {'symmetric_instances':1}),
                ('knob', {'symmetric_instances':3}),
                ('hands', {'symmetric_instances':3}),
            ]
            list_edges = [
                ('base', 'knob', {'motion_type':'SO2','all_DOFs':1}),
                ('base', 'hands', {'motion_type':'SO2','all_DOFs':1}),
            ]

    list_full_nodes, list_full_edges = graph_repeated_independent_extension(list_nodes, list_edges)
    G.add_nodes_from(list_full_nodes)
    G.add_edges_from(list_full_edges)
    nodes = list(G.nodes)

    G_out_nodes_trimmed = G.copy()
    G_out = G_out_nodes_trimmed.copy()
    return G_out

In [ ]:

def get_motion_category_text_labels_graph(category_name, crop_identical_instances=True, remove_invisible = False,):
    G = nx.DiGraph()
    G.name = category_name

    if category_name in set_rigid_stimuli:
        list_nodes = [
        ('base', {'motion_parts_visibility': 1, 'motion_type': 'None', 'all_DOFs':0, 'softness':0,  'symmetric_instances':1})
        ]
        list_edges = [
        ]


    elif category_name in set_soft_stimuli:
        if category_name in [  'nest_13s.jpg', 'hula_hoop_16s.jpg', 'chest1_14s.jpg', ]:
            list_nodes = [
            ('base', {'motion_parts_visibility': 1, 'motion_type': 'None', 'all_DOFs':0, 'softness':1,  'symmetric_instances':1}),
            ]

            list_edges = [
            ]

        elif category_name in ['whip_14s.jpg', 'wig_13s.jpg' ]:

            list_nodes = [
            ('base', {'motion_parts_visibility': 1, 'motion_type': 'E3', 'all_DOFs':6, 'softness':2,  'symmetric_instances':1}),
            ]
            list_edges = [
            ('base', 'base', ),
            ]

        elif category_name in ['beachball_16s.jpg', 'lasagna_13s.jpg', 'dough_19s.jpg', ]:

            list_nodes = [
            ('base', {'motion_parts_visibility': 1, 'motion_type': 'E3', 'all_DOFs':6, 'softness':1,  'symmetric_instances':1}),
            ]
            list_edges = [
            ('base', 'base', ),
            ]

        elif category_name in [ 'kimono_14s.jpg', 'mosquito_net_13s.jpg', 't-shirt_13s.jpg',  'tent_14n.jpg',]:
            list_nodes = [
            ('base', {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':2,  'symmetric_instances':1}),
            ]
            list_edges = [
            ('base', 'base', ),
            ]

    elif category_name in set_animal_stimuli:

        if category_name == 'starfish_15n.jpg':
            list_nodes = [
            ('tentacles_to_body', {'motion_parts_visibility': 1, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':1,  'symmetric_instances':5}),
            ('mouth', {'motion_parts_visibility': 0, 'motion_type': 'R', 'all_DOFs':1, 'softness':1,  'symmetric_instances':1}),

            ]
            list_edges = [
            ('mouth', 'mouth', ),
            ('tentacles_to_body', 'tentacles_to_body', ),
            ]


        if category_name == 'alligator_14n.jpg':
            list_nodes = [
                ('tail_to_body_motion',  {'motion_parts_visibility': 0, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':1,  'symmetric_instances':1}),
                ('jaws_to_head_motion',  {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':0,  'symmetric_instances':1}),
                ('limb_to_body_motion',  {'motion_parts_visibility': 1, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':1,  'symmetric_instances':4}),
                ('head_to_body_motion', {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':0,  'symmetric_instances': 1}),
                ]

            list_edges = [
                ('jaws_to_head_motion', 'jaws_to_head_motion',   ),
                ('tail_to_body_motion', 'tail_to_body_motion',   ),
                ('head_to_body_motion', 'head_to_body_motion', ),
                ('limb_to_body_motion', 'limb_to_body_motion',   ),

                ]
        elif category_name == 'beaver_13n.jpg':
            list_nodes = [
                ('head_to_body_motion',  {'motion_parts_visibility': 1, 'motion_type': 'SO3', 'all_DOFs':3, 'softness':0,  'symmetric_instances':1}),
                ('tail_to_body_motion', {'motion_parts_visibility': 1, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':2,  'symmetric_instances':1}),
                ('arms_motion',  {'motion_parts_visibility': 1, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':0,  'symmetric_instances':2}),
                ('legs_motion',  {'motion_parts_visibility': 1, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':0,  'symmetric_instances':2}),
                ]

            list_edges = [
                ('head_to_body_motion', 'head_to_body_motion',   ),
                ('tail_to_body_motion', 'tail_to_body_motion', ),

                ('arms_motion', 'arms_motion', ),
                ('legs_motion', 'legs_motion', ),

                ]
        elif category_name == 'chipmunk_16n.jpg':
            list_nodes = [
                ('head_to_body_motion',  {'motion_parts_visibility': 1, 'motion_type': 'SO3', 'all_DOFs':3, 'softness':0,  'symmetric_instances':1}),
                ('tail_to_body_motion', {'motion_parts_visibility': 1, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':2,  'symmetric_instances':1}),
                ('arms_motion',  {'motion_parts_visibility': 1, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':0,  'symmetric_instances':2}),
                ('legs_motion',  {'motion_parts_visibility': 1, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':0,  'symmetric_instances':2}),
                ]

            list_edges = [
                ('head_to_body_motion', 'head_to_body_motion',   ),
                ('tail_to_body_motion', 'tail_to_body_motion', ),

                ('arms_motion', 'arms_motion', ),
                ('legs_motion', 'legs_motion', ),

                ]

        elif category_name == 'cow_16n.jpg':
            list_nodes = [
                ('tail_to_body_motion',  {'motion_parts_visibility': 0, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':2,  'symmetric_instances':1}),
                ('head_to_body_motion',  {'motion_parts_visibility': 1, 'motion_type': 'SO3', 'all_DOFs':3, 'softness':0,  'symmetric_instances':1}),
                ('limb_to_body_motion',  {'motion_parts_visibility': 1, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':0,  'symmetric_instances':4}),
                ('knee_motion', {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':0,  'symmetric_instances':4}),
                ]

            list_edges = [
                ('head_to_body_motion', 'head_to_body_motion',   ),
                ('tail_to_body_motion', 'tail_to_body_motion',   ),
                ('limb_to_body_motion', 'knee_motion',   ),
                ('knee_motion', 'limb_to_body_motion', ),
                ('limb_to_body_motion', 'limb_to_body_motion',   ),
                ('knee_motion', 'knee_motion', ),

                ]
        elif category_name == 'hippopotamus_16s.jpg':
            list_nodes = [
                ('head_to_body_motion',  {'motion_parts_visibility': 1, 'motion_type': 'SO3', 'all_DOFs':3, 'softness':0,  'symmetric_instances':1}),

                ('jaws_relative_motion', {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':0,  'symmetric_instances':2}),

                ('limb_to_body_motion',  {'motion_parts_visibility': 1, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':1,  'symmetric_instances':4}),

                ]

            list_edges = [
                ('head_to_body_motion', 'head_to_body_motion',   ),
                ('jaws_relative_motion', 'jaws_relative_motion', ),

                ('limb_to_body_motion', 'limb_to_body_motion',   ),


                ]
        elif category_name == 'horse_17s.jpg':
            list_nodes = [
                ('tail_to_body_motion',  {'motion_parts_visibility': 1, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':2,  'symmetric_instances':1}),
                ('head_to_body_motion',  {'motion_parts_visibility': 1, 'motion_type': 'SO3', 'all_DOFs':3, 'softness':0,  'symmetric_instances':1}),
                ('limb_to_body_motion',  {'motion_parts_visibility': 1, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':0,  'symmetric_instances':4}),
                ('knee_motion', {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':0,  'symmetric_instances':4}),
                ]

            list_edges = [
                ('head_to_body_motion', 'head_to_body_motion',   ),
                ('tail_to_body_motion', 'tail_to_body_motion',   ),
                ('limb_to_body_motion', 'knee_motion', ),
                ('knee_motion', 'limb_to_body_motion', ),
                ('knee_motion', 'knee_motion',   ),
                ('limb_to_body_motion', 'limb_to_body_motion', ),
                ]
        elif category_name == 'iguana_20s.jpg':
            list_nodes = [
                ('tail_to_body_motion',  {'motion_parts_visibility': 0, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':2,  'symmetric_instances':1}),
                ('head_to_body_motion',  {'motion_parts_visibility': 1, 'motion_type': 'SO3', 'all_DOFs':3, 'softness':0,  'symmetric_instances':1}),
                ('jaws_relative_motion', {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':0,  'symmetric_instances':2}),
                ('spikes_relative_motion', {'motion_parts_visibility': 1, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':1,  'symmetric_instances':5}),
                ('limb_to_body_motion',  {'motion_parts_visibility': 0, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':1,  'symmetric_instances':4}),
                ]

            list_edges = [
                ('head_to_body_motion', 'head_to_body_motion',   ),
                ('tail_to_body_motion', 'tail_to_body_motion',   ),
                ('jaws_relative_motion', 'jaws_relative_motion', ),
                ('limb_to_body_motion', 'limb_to_body_motion', ),
                ]
        elif category_name == 'monkey_18n.jpg':
            list_nodes = [
                ('head_to_body_motion',  {'motion_parts_visibility': 1, 'motion_type': 'SO3', 'all_DOFs':3, 'softness':0,  'symmetric_instances':1}),
                ('tail_to_body_motion', {'motion_parts_visibility': 1, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':2,  'symmetric_instances':1}),

                ('wrist_to_arm_motion',  {'motion_parts_visibility': 0, 'motion_type': 'SO3', 'all_DOFs':3, 'softness':1,  'symmetric_instances':2}),
                ('elbow_motion', {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':0,  'symmetric_instances':2}),
                ('arms_motion',  {'motion_parts_visibility': 1, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':0,  'symmetric_instances':2}),

                ('legs_motion',  {'motion_parts_visibility': 1, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':0,  'symmetric_instances':2}),
                ('knee_motion', {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':0,  'symmetric_instances':2}),
                ]

            list_edges = [
                ('head_to_body_motion', 'head_to_body_motion',   ),
                ('tail_to_body_motion', 'tail_to_body_motion', ),

                ('wrist_to_arm_motion', 'wrist_to_arm_motion',   ),
                ('arms_motion', 'arms_motion', ),
                ('legs_motion', 'legs_motion', ),
                ('elbow_motion', 'elbow_motion', ),
                ('knee_motion', 'knee_motion', ),

                ('knee_motion', 'legs_motion',   ),
                ('legs_motion', 'knee_motion',   ),

                ('wrist_to_arm_motion', 'elbow_motion',  ),
                ('elbow_motion', 'wrist_to_arm_motion',  ),

                ('elbow_motion', 'arms_motion',   ),
                ('arms_motion', 'elbow_motion',  ),
                ]
        elif category_name == 'rabbit_13n.jpg':
            list_nodes = [
                ('head_to_body_motion',  {'motion_parts_visibility': 1, 'motion_type': 'SO3', 'all_DOFs':3, 'softness':0,  'symmetric_instances':1}),
                ('arms_motion',  {'motion_parts_visibility': 1, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':0,  'symmetric_instances':2}),
                ('legs_motion',  {'motion_parts_visibility': 1, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':0,  'symmetric_instances':2}),
                ]

            list_edges = [
                ('head_to_body_motion', 'head_to_body_motion',   ),
                ('arms_motion', 'arms_motion', ),
                ('legs_motion', 'legs_motion', ),
                ]
        elif category_name == 'butterfly_16n.jpg':
            list_nodes = [
                ('wing_rot', {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':0,  'symmetric_instances':1}),
                ('legs_rot', {'motion_parts_visibility': 1, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':1,  'symmetric_instances':5}),
            ]
            list_edges = [
                ('wing_rot', 'wing_rot', ),
                ('legs_rot', 'legs_rot', ),
            ]

        elif category_name == 'dragonfly_13s.jpg':

            list_nodes = [
                ('wing_rot', {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':0,  'symmetric_instances':2}),
                ('legs_rot', {'motion_parts_visibility': 1, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':1,  'symmetric_instances':5}),
            ]
            list_edges = [
                ('wing_rot', 'wing_rot', ),
                ('legs_rot', 'legs_rot', ),
            ]
        elif category_name == 'wasp_15n.jpg':

            list_nodes = [
                ('wing_rot', {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':0,  'symmetric_instances':1}),
                ('legs_rot', {'motion_parts_visibility': 1, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':1,  'symmetric_instances':5}),
            ]
            list_edges = [
                ('wing_rot', 'wing_rot', ),
                ('legs_rot', 'legs_rot', ),
            ]
    elif category_name in set_complex_stimuli:

        if category_name == 'crank_16s.jpg':
            list_nodes = [
                ('handle_rot',  {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':0,  'symmetric_instances':1}),
                ('arm_rot',     {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':0,  'symmetric_instances':1}),
                ('handle_disp', {'motion_parts_visibility': 1, 'motion_type': 'R', 'all_DOFs':1, 'softness':0,  'symmetric_instances':1}),
            ]
            list_edges = [
                ('arm_rot', 'arm_rot',   ),
                ('arm_rot', 'handle_disp',       ),
                ('handle_disp', 'arm_rot',       ),
                ('handle_rot', 'handle_rot', ),
            ]
        elif category_name == 'drawer_13s.jpg':
            list_nodes = [
                ('drawer_disp', {'motion_parts_visibility': 1, 'motion_type': 'R', 'all_DOFs':1, 'softness':0,  'symmetric_instances':3}),
            ]
            list_edges = [
                ('drawer_disp', 'drawer_disp'),
            ]


        elif category_name == 'easel_15s.jpg':
            list_nodes = [
                ('base_to_leg_rot', {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':0,  'symmetric_instances':1}),

            ]
            list_edges = [
                ('base_to_leg_rot', 'base_to_leg_rot'),

            ]

        elif category_name == 'helicopter_25s.jpg':
            list_nodes = [
                ('blade1_to_frame', {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':0,  'symmetric_instances':1}),
                ('blade2_to_frame',  {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':0,  'symmetric_instances':1}),
            ]
            list_edges = [
                ('blade1_to_frame', 'blade1_to_frame',),
                ('blade2_to_frame', 'blade2_to_frame',),
            ]

        elif category_name == 'joystick_14s.jpg':
            list_nodes = [
                ('joystick_to_base', {'motion_parts_visibility': 1, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':0,  'symmetric_instances':1}),
                ('buttons_to_base', {'motion_parts_visibility': 1, 'motion_type': 'R', 'all_DOFs':1, 'softness':0,  'symmetric_instances':2}),
            ]
            list_edges = [
                ('joystick_to_base', 'joystick_to_base', ),
                ('buttons_to_base', 'buttons_to_base', ),
            ]

        elif category_name == 'typewriter_13s.jpg':
            list_nodes = [
                ('key_press', {'motion_parts_visibility': 1, 'motion_type': 'R', 'all_DOFs':1, 'softness':0,  'symmetric_instances':5}),
            ]
            list_edges = [
                ('key_press', 'key_press', ),
            ]

        elif category_name == 'piano_13n.jpg':
            list_nodes = [
                ('key_press', {'motion_parts_visibility': 1, 'motion_type': 'R', 'all_DOFs':1, 'softness':0,  'symmetric_instances':5}),
                ('piano_lid_rot', {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':0,  'symmetric_instances':1}),
            ]
            list_edges = [
                ('key_press', 'key_press', ),
                ('piano_lid_rot', 'piano_lid_rot', ),
            ]

        elif category_name == 'bike_14s.jpg':
            list_nodes = [
                ('wheel_to_frame_rot', {'motion_parts_visibility': 1, 'motion_type': 'SO2+R', 'all_DOFs':1, 'softness':0,  'symmetric_instances':2}),
                ('pedals_rot', {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':0,  'symmetric_instances':1}),
            ]
            list_edges = [
                ('pedals_rot', 'wheel_to_frame_rot', ),
            ]

        elif category_name == 'blind_19s.jpg':
            list_nodes = [
                ('blind_motion', {'motion_parts_visibility': 1, 'motion_type': 'R', 'all_DOFs':1, 'softness':0,  'symmetric_instances':1}),
            ]
            list_edges = [
                ('blind_motion', 'blind_motion', ),
            ]


        elif category_name == 'bulldozer_22n.jpg':

            list_nodes = [
                ('links_to_base',  {'motion_parts_visibility': 1, 'motion_type': 'SO2+R', 'all_DOFs':1, 'softness':0,  'symmetric_instances':2}),

            ]
            list_edges = [
                ('links_to_base', 'links_to_base', ),
            ]

        elif category_name in ['watch_13s.jpg']:
            list_nodes = [
            ('knob_rotation', {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':0,  'symmetric_instances':3}),
            ('hands_rotation', {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':0,  'symmetric_instances':3}),
            ]
            list_edges = [
            ('knob_rotation', 'hands_rotation', ),
            ('hands_rotation', 'hands_rotation', ),
            ]

    elif category_name in set_complex_soft_non_animal_stimuli:
        if category_name == 'clipboard_14s.jpg':

            list_nodes = [
                ('clip_to_board', {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':0,  'symmetric_instances':1}),
                ('paper_rotation', {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':1,  'symmetric_instances':1}),

            ]
            list_edges = [
                ('clip_to_board', 'clip_to_board', ),
                ('paper_rotation', 'paper_rotation', ),
            ]

        elif category_name == 'umbrella_13s.jpg':
            list_nodes = [
            ('canopy_to_base', {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':0,  'symmetric_instances':5}),
            ('cloth_to_base', {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':1,  'symmetric_instances':5}),
            ('runner_to_base', {'motion_parts_visibility': 0, 'motion_type': 'R', 'all_DOFs':1, 'softness':0,  'symmetric_instances':1}),
            ]

            list_edges = [
            ('runner_to_base', 'canopy_to_base',),
            ('canopy_to_base', 'cloth_to_base', ),
            ]

        elif category_name in ['headlamp_14s.jpg',]:
            list_nodes = [
            ('knob_rotation', {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':0,  'symmetric_instances':1}),
            ('belt_motion', {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':1,  'symmetric_instances':1}),
            ]
            list_edges = [
            ('knob_rotation', 'knob_rotation', ),
            ('belt_motion', 'belt_motion', ),
            ]

        elif category_name == 'pacifier_14s.jpg':
            list_nodes = [
                ('ring_to_base', {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':0,  'symmetric_instances':1}),
                ('the_pacifier_musher', {'motion_parts_visibility': 1, 'motion_type': 'None', 'all_DOFs':0, 'softness':1,  'symmetric_instances':1}),

            ]
            list_edges = [
                ('ring_to_base', 'ring_to_base', ),
                ('the_pacifier_musher', 'the_pacifier_musher', ),
            ]

        elif category_name in ['quill_15s.jpg',]:
            list_nodes = [
            ('base_deform', {'motion_parts_visibility': 1, 'motion_type': '2SO2',  'all_DOFs':2, 'softness':1,  'symmetric_instances':1}),
            ('feather_to_base', {'motion_parts_visibility': 1, 'motion_type': 'SO2',  'all_DOFs':1, 'softness':1,  'symmetric_instances':1}),
            ]
            list_edges = [
            ('base_deform', 'base_deform', ),
            ('feather_to_base', 'feather_to_base', ),
            ]


        elif category_name in [ 'banana_13s.jpg', ]:
            list_nodes = [
            ('peel_to_base', {'motion_parts_visibility': 1, 'motion_type': 'E3', 'all_DOFs':6, 'softness':2,  'symmetric_instances':1}),
            ]
            list_edges = [
            ('peel_to_base', 'peel_to_base', ),
            ]


        elif category_name in ['pear_13s.jpg', ]:
            list_nodes = [
            ('base', {'motion_parts_visibility': 1, 'motion_type': 'SO3', 'all_DOFs':3, 'softness':1,  'symmetric_instances':1}),
            ]
            list_edges = [
            ('base', 'base', ),
            ]

        elif category_name in ['dress_13s.jpg', ]:
            list_nodes = [
            ('base_to_sleeves', {'motion_parts_visibility': 1, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':2,  'symmetric_instances':2}),
            ('base_motion_deform', {'motion_parts_visibility': 1, 'motion_type': '2SO2', 'all_DOFs':2, 'softness':2,  'symmetric_instances':2}),
            ]
            list_edges = [
            ('base_to_sleeves', 'base_to_sleeves', ),
            ('base_motion_deform', 'base_motion_deform', ),
            ]
        elif category_name in ['brace_14s.jpg',]:
            list_nodes = [
            ('brace_motion', {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':2,  'symmetric_instances':1}),
            ('ankle_motion', {'motion_parts_visibility': 1, 'motion_type': 'SO2', 'all_DOFs':1, 'softness':0,  'symmetric_instances':1}),
            ]
            list_edges = [
            ('brace_motion', 'ankle_motion', ),
            ]

    if crop_identical_instances:
        list_full_nodes, list_full_edges = list_nodes, list_edges
    else:
        list_full_nodes, list_full_edges = graph_repeated_independent_extension(list_nodes, list_edges)
    G.add_nodes_from(list_full_nodes)
    G.add_edges_from(list_full_edges)
    nodes = list(G.nodes)
    if remove_invisible:
        for node in nodes:
            if (G.nodes[node]['motion_parts_visibility']) == 0:
                G.remove_node(node)

    G_out_nodes_trimmed = G.copy()
    G_out = G_out_nodes_trimmed.copy()
    return G_out


In [ ]:
def plot_motion_graph(graph, type_graph ='text_labels_GMAG', animal_stimulus=False):
    if not animal_stimulus:
        size = len(graph.nodes)
        if size == 1:
            figsize = [3, 3.5]
        elif size == 2:
            figsize = [10, 8]
        else:
            figsize = [13, 10]
    else:
        figsize = [20,18]

    fig, ax = plt.subplots(figsize=figsize)
    pos = nx.circular_layout(graph)
    labels = dict()
    edge_labels = dict()


    if type_graph == 'text_labels_GMAG':
        for node in graph.nodes:
            labels[node] = (f"{node}\ntype: {graph.nodes[node]['motion_type']}\n"
                            f"softness: {graph.nodes[node]['softness']}\n"
                            )

    else:
        for node in graph.nodes:
            labels[node] = node
        for n1, n2 in graph.edges:
            edge_labels[(n1, n2)] = (f"{graph.edges[(n1, n2)]['motion_type']}\n"
                                     )

    for i in pos:
        pos[i] = np.dot(np.array([[1, -1], [1, 1]]), pos[i])
    nx.draw_networkx_nodes(graph, pos, ax=ax, node_color='white')
    for i in pos:
        pos[i] /= 1.1
    nx.draw_networkx_labels(graph, pos, labels=labels, ax=ax)

    for i in pos:
        pos[i] /= 1.3
        pos[i] -= [0, (0.05/len(pos))]

    nx.draw_networkx_nodes(graph, pos, ax=ax)
    if type_graph != 'traditional':
        nx.draw_networkx_edges(graph, pos, ax=ax, arrowsize=20+(20/(1+abs(len(pos)))), width=1)
    else:
        nx.draw_networkx_edges(graph, pos, ax=ax, width=1, label=True)

        nx.draw_networkx_edge_labels(
        graph, pos,
        edge_labels,
        clip_on=True)

    nx.draw(graph, with_labels=False, ax=ax, pos=pos)
    plt.axis('off')
    plt.show()

In [ ]:
def Graph_model_RDM(list_used1, list_used2, graph_generator=get_motion_category_text_labels_graph, similarity=get_text_labels_MAG_similarity, upper_triangular=False):

    output_df = pd.DataFrame(index=list_used1, columns=list_used2)
    for i in range(len(list_used1)):
        for j in range(len(list_used2)):
            if upper_triangular:

                if i >= j:
                    print(i, j)
                    output_df.iloc[i, j] = similarity(graph_generator(list_used1[i], add_symmetric=True), graph_generator(list_used2[j], add_symmetric=True))

            else:

                output_df.iloc[i, j] = similarity(graph_generator(list_used1[i]), graph_generator(list_used2[j]))
            print( list_used1[i], list_used2[j] )
    if upper_triangular:
        output_df = output_df.fillna(output_df.T)

    return output_df

In [ ]:
all_lazy_stimuli_ext_asymmetric = Graph_model_RDM(list_ext, list_ext, get_motion_traditional_graph, get_traditional_similarity)

upper_tri_ext = (all_lazy_stimuli_ext_asymmetric.where(np.triu(np.ones(all_lazy_stimuli_ext_asymmetric.shape)).astype(np.bool)) +
                 all_lazy_stimuli_ext_asymmetric.where(np.tril(np.ones(all_lazy_stimuli_ext_asymmetric.shape)).astype(np.bool)).T)
all_lazy_stimuli_ext = upper_tri_ext.fillna(upper_tri_ext.T)/2 * (1-np.eye(len(upper_tri_ext)))

all_lazy_stimuli_ext.to_csv('exp_cond_trad_RDM.csv')

np.where(pd.isnull(all_lazy_stimuli_ext))

In [ ]:
all_lazy_stimuli_ext_asymmetric = Graph_model_RDM(list_ext, list_ext, get_motion_category_text_labels_graph, get_text_labels_MAG_similarity)

upper_tri_ext = (all_lazy_stimuli_ext_asymmetric.where(np.triu(np.ones(all_lazy_stimuli_ext_asymmetric.shape)).astype(np.bool)) +
                 all_lazy_stimuli_ext_asymmetric.where(np.tril(np.ones(all_lazy_stimuli_ext_asymmetric.shape)).astype(np.bool)).T)
all_lazy_stimuli_ext = upper_tri_ext.fillna(upper_tri_ext.T)/2 * (1-np.eye(len(upper_tri_ext)))

#all_lazy_stimuli_ext.to_csv('exp_cond_MAG_RDM.csv')
#all_lazy_stimuli_ext.to_csv('exp_cond_MAG_RDM_removed_duplicates.csv')
#all_lazy_stimuli_ext.to_csv('exp_cond_MAG_RDM_removed_invisible.csv')
np.where(pd.isnull(all_lazy_stimuli_ext))

In [ ]:
set_rigid_stimuli = {'ashtray_14n.jpg', 'axe_14n.jpg', 'boat_14n.jpg', 'bobsled_14s.jpg', 'candelabra_14s.jpg', 'coat_rack_13s.jpg', 'cufflink_16s.jpg', 'drain_16s.jpg', 'grate_13s.jpg', 'horseshoe_13n.jpg', 'hovercraft_13n.jpg', 'key_17s.jpg', 'television_14n.jpg', 'microscope_14n.jpg', }

set_soft_stimuli = {'beachball_16s.jpg',  'kimono_14s.jpg', 'lasagna_13s.jpg', 'mosquito_net_13s.jpg', 't-shirt_13s.jpg', 'tent_14n.jpg', 'whip_14s.jpg', 'wig_13s.jpg', 'hula_hoop_16s.jpg', 'dough_19s.jpg', 'nest_13s.jpg', 'chest1_14s.jpg',}

set_complex_stimuli = {'bike_14s.jpg', 'blind_19s.jpg', 'bulldozer_22n.jpg', 'crank_16s.jpg', 'helicopter_25s.jpg', 'joystick_14s.jpg', 'piano_13n.jpg', 'typewriter_13s.jpg', 'easel_15s.jpg', 'drawer_13s.jpg', 'watch_13s.jpg',}

set_complex_soft_non_animal_stimuli = {'umbrella_13s.jpg', 'pacifier_14s.jpg', 'clipboard_14s.jpg', 'headlamp_14s.jpg', 'banana_13s.jpg', 'pear_13s.jpg', 'quill_15s.jpg', 'brace_14s.jpg', 'dress_13s.jpg', }

set_animal_stimuli = {'alligator_14n.jpg', 'beaver_13n.jpg', 'butterfly_16n.jpg', 'chipmunk_16n.jpg', 'cow_16n.jpg', 'dragonfly_13s.jpg', 'hippopotamus_16s.jpg', 'horse_17s.jpg', 'iguana_20s.jpg', 'monkey_18n.jpg', 'rabbit_13n.jpg', 'starfish_15n.jpg', 'wasp_15n.jpg',}

mypath = 'Repeated Stimuli'
onlyfiles = [f for f in listdir(mypath) if isfile(join(mypath, f))][:10]
from os import walk

f = []
for (dirpath, dirnames, filenames) in walk(mypath):
    f.extend(filenames)
    break
f = [f'{mypath}/{i}' for i in f]
len(f)
for i in f[:]:
    stim_name = (i[1+len(mypath):])
    print(stim_name)

    if stim_name in set_rigid_stimuli:
        print('Rigid object, INCLUDED')
    elif stim_name in set_soft_stimuli | set_complex_stimuli | set_complex_soft_non_animal_stimuli | set_animal_stimuli:
        animal_stimulus = stim_name in set_animal_stimuli
        graph_traditional = get_motion_traditional_graph(stim_name)
        GMAG_complete = get_motion_category_text_labels_graph(stim_name, crop_identical_instances=False, remove_invisible=False)
        GMAG_no_duplicates = get_motion_category_text_labels_graph(stim_name, crop_identical_instances=True, remove_invisible=False)
        GMAG_no_invisibles = get_motion_category_text_labels_graph(stim_name, crop_identical_instances=False, remove_invisible=True)
        GMAG_no_invisibles_or_duplicates = get_motion_category_text_labels_graph(stim_name, crop_identical_instances=True, remove_invisible=True)
        if stim_name in set_soft_stimuli or stim_name in set_complex_soft_non_animal_stimuli:
            print("The traditional skeleton graph model cannot model the soft objects")
        else:
            print("Traditional skeleton graph model:")
            plot_motion_graph(graph_traditional, 'traditional', animal_stimulus=animal_stimulus)

        print("Complete GMAG model:")
        plot_motion_graph(GMAG_complete, animal_stimulus=animal_stimulus)
        print("GMAG model with excluded duplicate motions:")
        plot_motion_graph(GMAG_no_duplicates, animal_stimulus=animal_stimulus,)
        print("GMAG model with excluded invisible motions:")
        plot_motion_graph(GMAG_no_invisibles, animal_stimulus=animal_stimulus, )
        print("GMAG model with excluded duplicate and invisible motions:")
        plot_motion_graph(GMAG_no_invisibles_or_duplicates, animal_stimulus=animal_stimulus)

    else:
        print('EXCLUDED')
    print('Stimulus image:')
    img = mpimg.imread(i)
    imgplot = plt.imshow(img)
    plt.show()
    print()
